# Two-epoch coalescent inference with BFFG

Backward-Filtering Forward-Guiding (BFFG) inference of a two-epoch coalescent model, using sparse single-mutation observations (singletons, doubletons, ...).

**Model.** Block coalescent with $n$ samples and a piecewise-constant pairwise coalescence rate:
$$
\theta(t) = \begin{cases} \theta_1, & t < \tau \\ \theta_2, & t \ge \tau \end{cases}
$$
with epoch boundary $\tau$ assumed known. The pairwise rate is multiplied by $\binom{k}{2}$ when $k$ lineages remain.

**Observation model.** At each locus we observe one mutation type $j \in \{1,\dots,n-1\}$ (singleton/doubleton/...). The probability of observing type $j$ given the genealogy $g$ is proportional to the total branch length of type-$j$ lineages, $B_j(g)$. We use the **joint probability graph** to compute these probabilities and to sample paths conditional on the observed mutation type.

**BFFG.** Paths are sampled under a homogeneous proposal with constant rate $\theta_0$ via `sample_path_conditioned`. The importance weight corrects to the time-inhomogeneous target by re-weighting using the density of coalescence events in each epoch:
$$
\log w = \sum_k \bigl[\log \theta(t_k) - \log \theta_0\bigr]
       - \sum_{\text{epoch }e} (\theta_e - \theta_0)\, P_e
$$
where $t_k$ are coalescence times and $P_e = \sum_{\text{sojourns in }e} \binom{k_s}{2}\,\Delta t_s$ is the pair-time spent in epoch $e$.

**Inference.** MCMC over $(\theta_1, \theta_2)$ in two ways:

1. A **manual, transparent** BFFG log-likelihood plumbed into `MCMC(log_prob_fn=...)`. Easy to introspect for diagnostics — used through section 9.
2. The **package's `bffg_log_prob(..., return_model=True)`** plumbed into `MCMC(model=..., likelihood_correction=...)`, which uses the FFI-cached sojourn-time computation for the model term and the exit-rate-ratio importance weight for the correction. Section 10.

In [1]:
from phasic import (
    Graph, MCMC, GaussPrior, with_ipv,
    StateIndexer, Property,
    path_to_rewards, path_exit_rates,
    importance_log_weight_from_rates, importance_weighted_log_likelihood,
)

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from jax.scipy.special import logsumexp

from vscodenb import set_vscode_theme
set_vscode_theme()

np.random.seed(42)
rng = np.random.default_rng(42)

## 1. Build the block coalescent graph

State vector: $(b_1, b_2, \dots, b_n)$ where $b_j$ is the number of blocks of size $j$. A coalescence between a size-$i$ block and a size-$j$ block produces a size-$(i+j)$ block; the rate is the pair count times the per-pair coalescence rate $\theta_1$.

We expose $\theta_1$ as a single parameterized coefficient — the resulting graph has constant rate. Time-inhomogeneity is added later through the BFFG importance weight.

In [2]:
from itertools import combinations_with_replacement

N_SAMPLES = 4   # singletons, doubletons, tripletons → 3 mutation types

# Build the indexer first so the initial state vector can be constructed via props_to_index.
indexer = StateIndexer(
    lineages=[Property('ton', min_value=1, max_value=N_SAMPLES)]
)
ipv = [0] * indexer.state_length
ipv[indexer.props_to_index(ton=1)] = N_SAMPLES  # start with N_SAMPLES singletons

@with_ipv(ipv)
def coal_callback(state, indexer=None):
    transitions = []
    for i, j in combinations_with_replacement(range(state.size), 2):
        same = int(i == j)
        if same and state[i] < 2:
            continue
        if not same and (state[i] < 1 or state[j] < 1):
            continue
        new = state.copy()
        new[i] -= 1
        new[j] -= 1
        new[min(i + j + 1, state.size - 1)] += 1
        pair_count = state[i] * (state[j] - same) / (1 + same)
        transitions.append([new, [pair_count]])
    return transitions

g_cont = Graph(coal_callback, indexer=indexer)
g_cont.update_weights([1.0])  # placeholder; proposal rate set below
print(f"Continuous coalescent graph: {g_cont.vertices_length()} vertices, "
      f"state length {g_cont.state_length()}")

Continuous coalescent graph: 6 vertices, state length 4


## 2. Build the joint probability graph

The joint probability graph extends the state vector with a per-mutation-type counter and adds mutation-rate edges. The terminal vertices represent the observation "the mutation that occurred was on a type-$j$ branch". The probabilities of these terminals — given $\theta$ and the mutation rate $\mu$ — give the categorical observation likelihood.

In [3]:
MUTATION_RATE = 0.01

# tot_reward_limit=1 → exactly one mutation per locus (the sparse single-mutation observation).
# The same indexer is reused: joint_prob_graph extends it internally with reward properties.
jg_cont = g_cont.joint_prob_graph(
    indexer,
    mutation_rate=MUTATION_RATE,
    reward_limit=8,
)
jg_cont.is_discrete = False  # for path sampling we want continuous time

jg_disc = g_cont.joint_prob_graph(
    indexer,
    mutation_rate=MUTATION_RATE,
    reward_limit=1,
)
# jg_disc keeps is_discrete=True (set by joint_prob_graph) for joint_prob_table()

print(f"Joint graph: {jg_cont.vertices_length()} vertices, "
      f"state length {jg_cont.state_length()}, "
      f"param_length {jg_cont.param_length()}")

Joint graph: 1633 vertices, state length 8, param_length 2


In [4]:
# Proposal parameters
THETA_0 = 1.0                          # constant proposal rate
theta_proposal = np.array([THETA_0, MUTATION_RATE])

jg_disc.update_weights(theta_proposal.tolist())
jpt = jg_disc.joint_prob_table()
print(jpt)

# Each row of joint_prob_table corresponds to a t-vertex (SFS bin). The columns
# before 'prob' encode the reward state (counts of each mutation type).
# Build a map: mutation_type (1..n-1) → t-vertex index.
terminal_indices = list(jpt.index)
term_state_cols = jpt.columns[:-1]
print("Reward state columns:", list(term_state_cols))

mut_type_to_terminal = {}
for tv in terminal_indices:
    state_row = jpt.loc[tv, term_state_cols].to_numpy(dtype=int)
    if state_row.sum() == 1:
        # column position of the 1 → block size of the mutated branch
        j = int(np.argmax(state_row)) + 1
        mut_type_to_terminal[j] = int(tv)
print("Mutation type → t-vertex index:", mut_type_to_terminal)

# We also need to map from path['vertex_indices'][-2] (the last non-absorbing vertex) back
# to the t-vertex set. Each t-vertex either IS the penultimate path vertex or the path
# went absorbing→absorbing — we just need the set of valid t-vertices for fast lookup.
valid_t_vertices = set(terminal_indices)

                ton_1  ton_2  ton_3  ton_4          prob
t_vertex_index                                          
8                   0      0      0      0  9.996334e-01
15                  0      1      0      0  9.994669e-05
16                  1      0      0      0  1.999023e-04
18                  0      0      1      0  6.662890e-05
22                  1      0      1      0  2.220623e-08
23                  1      1      0      0  1.776682e-08
25                  0      1      1      0  2.220741e-09
26                  1      1      1      0  8.881683e-13
Reward state columns: ['ton_1', 'ton_2', 'ton_3', 'ton_4']
Mutation type → t-vertex index: {2: 15, 1: 16, 3: 18}


## 3. Define the time-inhomogeneous target

Two-epoch piecewise-constant rate with hard switch at `EPOCH_BOUNDARY`. `theta_target_fn(theta_mcmc, t)` returns the per-pair rate at time $t$ — this is the function BFFG will use to re-weight sampled paths.

In [5]:
EPOCH_BOUNDARY = 0.5
TRUE_THETA1 = 2.0   # epoch 1: faster coalescence (smaller Ne)
TRUE_THETA2 = 0.5   # epoch 2: slower coalescence (larger Ne)
true_theta = np.array([TRUE_THETA1, TRUE_THETA2])

def theta_target_fn(theta_mcmc, t):
    # theta_mcmc = [theta1, theta2]; returns scalar rate at time t
    return jnp.where(t < EPOCH_BOUNDARY, theta_mcmc[0], theta_mcmc[1])

def theta_target_np(theta_np, t):
    return theta_np[0] if t < EPOCH_BOUNDARY else theta_np[1]

## 4. Importance weight from event density

For a coalescent path with coalescence times $t_1 < t_2 < \dots$ and lineage counts $k_1 > k_2 > \dots$, the proposal density is
$$
p_0(\text{path}) = \prod_k \theta_0 \binom{k_s}{2} \cdot \exp\!\left[-\theta_0 \sum_s \binom{k_s}{2}\,\Delta t_s\right]
$$
and the target density replaces $\theta_0$ with $\theta(t_k)$ at the event and integrates $\theta(t)\binom{k_s}{2}$ over each sojourn. The combinatorial factors cancel in the ratio, leaving
$$
\log w = \sum_k \bigl[\log \theta(t_k) - \log \theta_0\bigr] - \sum_e (\theta_e - \theta_0) P_e
$$
where $P_e$ is the pair-time accumulated in epoch $e$. We compute this directly from the path sampled on the joint graph.

In [6]:
def path_to_epoch_quantities(jg, path, n_samples, epoch_boundary):
    """Extract per-epoch coalescence event times and pair-time integrals.

    Walks the path skipping the start vertex (block_sum=0, t=0) and breaks
    when only one lineage remains (no further coalescence possible) — this
    avoids polluting the sums with the artificial t-vertex → absorbing edge
    or trips through the trash loop.

    Returns
    -------
    event_times_per_epoch : list of np.ndarray
        Coalescence event times falling in each epoch.
    pair_time_per_epoch : np.ndarray, shape (2,)
        Σ C(k,2) Δt accumulated in each epoch.
    """
    indices = path['vertex_indices']
    times = path['entry_times']
    states = jg.states()
    block_slice = slice(0, n_samples)

    event_times_e1, event_times_e2 = [], []
    pair_time = np.zeros(2)

    # step 0 is the starting vertex (state all zeros, t=0); skip it.
    for step in range(1, len(indices) - 1):
        v_now = int(indices[step])
        v_next = int(indices[step + 1])
        n_blocks = int(states[v_now, block_slice].sum())
        if n_blocks < 2:
            # last lineage absorbed — any further transition is the artificial
            # t-vertex → absorbing edge (or trash loop), not part of the coalescent
            break
        t_enter = times[step]
        t_exit = times[step + 1]
        dt = t_exit - t_enter
        pair_count = n_blocks * (n_blocks - 1) / 2.0
        if t_exit <= epoch_boundary:
            pair_time[0] += pair_count * dt
        elif t_enter >= epoch_boundary:
            pair_time[1] += pair_count * dt
        else:
            pair_time[0] += pair_count * (epoch_boundary - t_enter)
            pair_time[1] += pair_count * (t_exit - epoch_boundary)

        # coalescence: total block count drops by 1 (mutation transitions leave it unchanged)
        n_next = int(states[v_next, block_slice].sum())
        if n_next == n_blocks - 1:
            if t_exit < epoch_boundary:
                event_times_e1.append(t_exit)
            else:
                event_times_e2.append(t_exit)

    return ([np.array(event_times_e1), np.array(event_times_e2)], pair_time)


def log_importance_weight(epoch_events, pair_time, theta_target, theta_0):
    """BFFG log importance weight for a two-epoch homogeneous-proposal path."""
    log_w = 0.0
    for e, ev_times in enumerate(epoch_events):
        n_events = len(ev_times)
        log_w += n_events * (np.log(theta_target[e]) - np.log(theta_0))
        log_w -= (theta_target[e] - theta_0) * pair_time[e]
    return log_w

## 5. Simulate observations under the true two-epoch model

We simulate sparse single-mutation observations under the **true** time-inhomogeneous model by importance-weighted rejection sampling: propose a path under the homogeneous proposal, accept with probability $\min(1, w/M)$ where $M$ is a (rough) bound on $w$. Each accepted path's terminal vertex tells us which SFS bin the mutation fell into.

In [11]:
def path_terminal_t_vertex(path, valid_t_vertices):
    """Return the t-vertex (SFS bin) the path landed on, or None if it went to trash."""
    indices = path['vertex_indices']
    # Walk back from the absorbing vertex to find the last t-vertex.
    for k in range(len(indices) - 1, -1, -1):
        v = int(indices[k])
        if v in valid_t_vertices:
            return v
    return None


def simulate_observation(jg, theta_proposal, true_theta, theta_0,
                         epoch_boundary, n_samples, valid_t_vertices,
                         max_tries=1000):
    """Simulate one observation under the time-inhomogeneous target.

    Returns (t_vertex_index, log_weight_used).
    """
    jg.update_weights(theta_proposal.tolist())
    log_M = 5.0
    for _ in range(max_tries):
        path = jg.sample_path()
        epoch_events, pair_time = path_to_epoch_quantities(
            jg, path, n_samples, epoch_boundary
        )
        log_w = log_importance_weight(epoch_events, pair_time, true_theta, theta_0)
        if np.log(rng.uniform()) < log_w - log_M:
            t_v = path_terminal_t_vertex(path, valid_t_vertices)
            if t_v is None:
                continue  # went to trash, retry
            return t_v, log_w
    raise RuntimeError("Rejection sampler failed — increase max_tries or log_M.")


N_LOCI = 500
observed_data = np.empty(N_LOCI, dtype=np.int32)
for locus in range(N_LOCI):
    term, _ = simulate_observation(
        jg_cont, theta_proposal, true_theta, THETA_0,
        EPOCH_BOUNDARY, N_SAMPLES, valid_t_vertices,
    )
    observed_data[locus] = term

# Summary: how many of each mutation type?
mut_type_counts = np.zeros(N_SAMPLES - 1, dtype=int)
term_to_type = {v: k for k, v in mut_type_to_terminal.items()}
for v in observed_data:
    if int(v) in term_to_type:
        mut_type_counts[term_to_type[int(v)] - 1] += 1
print("Simulated SFS counts (singletons, doubletons, ...):", mut_type_counts)

RuntimeError: Rejection sampler failed — increase max_tries or log_M.

## 6. BFFG log-likelihood (manual, transparent version)

We define the BFFG log-likelihood explicitly so the diagnostic plots can introspect it. For each locus:

1. Sample $M$ paths conditioned on the observed terminal under the homogeneous proposal $\theta_0$.
2. Compute $\log w_m$ for each path under $\theta_{\text{target}}$.
3. Estimate $\log p(\text{obs}\mid\theta_{\text{target}}) = \log p(\text{obs}\mid\theta_0) + \log \hat E[w]$.

In [8]:
# Pre-compute terminal probabilities under the proposal (constant in MCMC)
jg_disc.update_weights(theta_proposal.tolist())
jpt_proposal = jg_disc.joint_prob_table()
log_p_obs_under_proposal = {
    int(tv): float(np.log(jpt_proposal.loc[tv, 'prob']))
    for tv in jpt_proposal.index
}


def bffg_log_likelihood(theta_mcmc, observed_data, n_paths=50, return_diagnostics=False):
    theta_np = np.asarray(theta_mcmc)
    total = 0.0
    diag = {'log_weights_per_locus': [], 'ess_per_locus': []}
    for locus in range(len(observed_data)):
        target_v = int(observed_data[locus])
        log_p0 = log_p_obs_under_proposal[target_v]
        log_weights = np.empty(n_paths)
        for m in range(n_paths):
            path = jg_cont.sample_path_conditioned([target_v])
            ev, pt = path_to_epoch_quantities(jg_cont, path, N_SAMPLES, EPOCH_BOUNDARY)
            log_weights[m] = log_importance_weight(ev, pt, theta_np, THETA_0)
        log_mean_w = float(logsumexp(jnp.array(log_weights)) - jnp.log(n_paths))
        total += log_p0 + log_mean_w
        if return_diagnostics:
            diag['log_weights_per_locus'].append(log_weights)
            w = np.exp(log_weights - log_weights.max())
            diag['ess_per_locus'].append((w.sum())**2 / (w**2).sum())
    return (total, diag) if return_diagnostics else total

## 7. Diagnostic plots before MCMC

Sanity-check the importance sampler: weight distributions, ESS per locus, log-likelihood surface near the truth.

> **Note on variance.** When the target rates `[θ₁, θ₂]` are far from the proposal rate `θ₀`, the importance weights have very high variance — the per-locus log-weight histogram below will show a long right tail. With small `n_paths` and many loci, this manifests as poor MCMC mixing (low acceptance, high R-hat). For realistic inference you would either pick a proposal closer to the bulk of the posterior, increase `n_paths` substantially (200+), or use a per-iteration adaptive proposal. Section 10 compares against the package's `bffg_log_prob` which uses a different (deterministic, seed-fixed) variance trade-off.

In [9]:
# Diagnostic 1: log-weight distributions and ESS at the true parameters
ll_true, diag = bffg_log_likelihood(true_theta, observed_data, n_paths=50, return_diagnostics=True)
print(f"BFFG log-likelihood at truth: {ll_true:.2f}")
all_log_w = np.concatenate(diag['log_weights_per_locus'])
ess = np.array(diag['ess_per_locus'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(all_log_w, bins=40, color='steelblue', edgecolor='white')
axes[0].set(xlabel='log w', ylabel='count', title='Log importance weights at truth (all loci)')
axes[1].hist(ess, bins=20, color='seagreen', edgecolor='white')
axes[1].axvline(50, color='k', ls='--', label='n_paths')
axes[1].set(xlabel='ESS', ylabel='count', title='Per-locus ESS at truth')
axes[1].legend()
plt.tight_layout(); plt.show()

KeyError: 0

In [ ]:
# Diagnostic 2: log-likelihood surface over (theta1, theta2)
grid = np.linspace(0.2, 4.0, 11)
Z = np.full((len(grid), len(grid)), np.nan)
for i, t1 in enumerate(grid):
    for j, t2 in enumerate(grid):
        Z[i, j] = bffg_log_likelihood(np.array([t1, t2]), observed_data, n_paths=20)

fig, ax = plt.subplots(figsize=(5.5, 5))
T1, T2 = np.meshgrid(grid, grid, indexing='ij')
im = ax.contourf(T1, T2, Z, levels=15, cmap='viridis')
ax.scatter([TRUE_THETA1], [TRUE_THETA2], color='red', marker='*', s=180, label='truth')
ax.set(xlabel=r'$\theta_1$ (epoch 1)', ylabel=r'$\theta_2$ (epoch 2)',
       title='BFFG log-likelihood surface')
plt.colorbar(im, ax=ax, label='log L')
ax.legend(); plt.tight_layout(); plt.show()

## 8. MCMC inference

Plug the BFFG log-likelihood into `MCMC` via the `log_prob_fn` mode (the cleanest route — `bffg_log_prob` from the package implements the same idea but uses an exit-rate ratio instead of the event-density form, and is geared toward the FFI/JIT path).

In [ ]:
def log_prob_fn(theta):
    return bffg_log_likelihood(theta, observed_data, n_paths=20)

mcmc = MCMC(
    log_prob_fn=log_prob_fn,
    prior=[GaussPrior(ci=[0.05, 8.0]), GaussPrior(ci=[0.05, 8.0])],
    theta_dim=2,
    n_samples=400,
    n_chains=2,
    burn_in=200,
    thin=1,
    proposal_scale=0.25,
    positive_params=True,
    jit=False,           # log_prob_fn calls back into Python/numpy
    seed=7,
    verbose=True,
    progress=True,
)
mcmc.run()
mcmc.summary()

## 9. Posterior diagnostics

In [ ]:
results = mcmc.get_results()
chains = results['chains']  # (n_chains, n_samples, 2)
samples = results['particles']  # (n_total, 2)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))

# trace plots
for c in range(chains.shape[0]):
    axes[0, 0].plot(chains[c, :, 0], lw=0.6, alpha=0.8)
    axes[0, 1].plot(chains[c, :, 1], lw=0.6, alpha=0.8)
axes[0, 0].axhline(TRUE_THETA1, color='red', ls='--')
axes[0, 1].axhline(TRUE_THETA2, color='red', ls='--')
axes[0, 0].set(title=r'Trace: $\theta_1$', xlabel='iter', ylabel=r'$\theta_1$')
axes[0, 1].set(title=r'Trace: $\theta_2$', xlabel='iter', ylabel=r'$\theta_2$')

# marginal posteriors
axes[1, 0].hist(samples[:, 0], bins=35, color='steelblue', edgecolor='white', density=True)
axes[1, 0].axvline(TRUE_THETA1, color='red', ls='--', label='truth')
axes[1, 0].set(xlabel=r'$\theta_1$', ylabel='density', title=r'Posterior $\theta_1$')
axes[1, 0].legend()

axes[1, 1].hist(samples[:, 1], bins=35, color='seagreen', edgecolor='white', density=True)
axes[1, 1].axvline(TRUE_THETA2, color='red', ls='--', label='truth')
axes[1, 1].set(xlabel=r'$\theta_2$', ylabel='density', title=r'Posterior $\theta_2$')
axes[1, 1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# joint posterior + log-likelihood contours
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.contour(T1, T2, Z, levels=10, cmap='Greys', alpha=0.6)
ax.scatter(samples[:, 0], samples[:, 1], s=4, alpha=0.25, color='steelblue', label='posterior')
ax.scatter([TRUE_THETA1], [TRUE_THETA2], color='red', marker='*', s=220,
           edgecolor='black', label='truth', zorder=5)
ax.set(xlabel=r'$\theta_1$', ylabel=r'$\theta_2$',
       title='Joint posterior over log-likelihood contours')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Per-chain autocorrelation
from numpy.fft import fft, ifft

def autocorr(x, max_lag=80):
    x = x - x.mean()
    n = len(x)
    f = fft(x, n=2*n)
    acf = ifft(f * np.conj(f)).real[:max_lag] / (n * x.var())
    return acf

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for c in range(chains.shape[0]):
    axes[0].plot(autocorr(chains[c, :, 0]), label=f'chain {c+1}')
    axes[1].plot(autocorr(chains[c, :, 1]), label=f'chain {c+1}')
for ax, name in zip(axes, [r'$\theta_1$', r'$\theta_2$']):
    ax.axhline(0, color='k', lw=0.5)
    ax.set(xlabel='lag', ylabel='autocorr', title=f'Autocorrelation: {name}')
    ax.legend()
plt.tight_layout(); plt.show()

print('R-hat:', results.get('rhat'))
print('ESS:  ', results.get('ess'))

## 10. Compare with the package's `bffg_log_prob`

The library ships a ready-made BFFG inference helper that returns a JIT-compatible model term plus a stochastic correction. It uses a **different importance-weight formulation** than ours:

- **Manual (sections 4–9):** event-density form, exploiting the special structure of homogeneous coalescent proposal vs piecewise-constant target. Requires us to know that the model is a coalescent.
- **Package (`bffg_log_prob`):** generic exit-rate-ratio form,
$$
\log w = \sum_{\text{transient }v} \bigl[\log r^{\text{tgt}}_v - \log r^{\text{prop}}_v\bigr] - (r^{\text{tgt}}_v - r^{\text{prop}}_v)\,s_v + \log\frac{p^{\text{tgt}}_{v\to v'}}{p^{\text{prop}}_{v\to v'}}
$$
which works for any phase-type graph but evaluates `theta_target_fn` at every transient vertex along the path.

The two estimators target the same posterior but have different variance profiles. We run MCMC with the package version and overlay both posteriors.

In [ ]:
from phasic import bffg_log_prob

# bffg_log_prob expects:
#   theta_proposal: GRAPH-level parameter vector (length = jg.param_length() = 2 here: [theta, mu])
#   theta_target_fn(theta_mcmc, t): returns a GRAPH-level vector of length param_length()
#       — the rate coefficient at time t plus the unchanged mutation rate.
#   theta_proposal_fn(theta_mcmc): JAX-traceable; returns the GRAPH-level vector used by
#       the model term (which is evaluated under the homogeneous proposal).

def theta_target_fn_pkg(theta_mcmc, t):
    """Graph-level theta vector at time t: [coalescence_rate(t), mutation_rate]."""
    rate = jnp.where(t < EPOCH_BOUNDARY, theta_mcmc[0], theta_mcmc[1])
    return jnp.array([rate, MUTATION_RATE])

def theta_proposal_fn_pkg(theta_mcmc):
    """Model term uses the homogeneous proposal regardless of theta_mcmc."""
    return jnp.array([THETA_0, MUTATION_RATE])

# Re-prime jg_disc to the proposal (joint_prob_table caching)
jg_disc.update_weights(theta_proposal.tolist())

model_pkg, correction_pkg = bffg_log_prob(
    jg_disc=jg_disc,
    jg_continuous=jg_cont,
    theta_proposal=theta_proposal,
    theta_target_fn=theta_target_fn_pkg,
    observed_data=observed_data,
    n_paths=20,
    theta_proposal_fn=theta_proposal_fn_pkg,
    return_model=True,
)

# Sanity check: evaluate at the truth
probs, _ = model_pkg(jnp.array(true_theta), jnp.array(observed_data, dtype=jnp.int32))
log_p_model_truth = float(jnp.sum(jnp.log(probs + 1e-30)))
log_corr_truth = float(correction_pkg(jnp.array(true_theta)))
print(f"Package BFFG at truth:  log P_model = {log_p_model_truth:.2f}, "
      f"log E[w] = {log_corr_truth:.2f}, total = {log_p_model_truth + log_corr_truth:.2f}")
print(f"Manual  BFFG at truth:  total = {ll_true:.2f}")

In [ ]:
# Run MCMC using the package's (model, likelihood_correction) split.
# The model term uses the FFI trace cache (JIT-compatible), the correction
# uses sampled paths (non-JIT). MCMC.likelihood_correction adds the
# correction outside the JIT'd model evaluation.

mcmc_pkg = MCMC(
    model=model_pkg,
    observed_data=jnp.array(observed_data, dtype=jnp.int32),
    likelihood_correction=correction_pkg,
    prior=[GaussPrior(ci=[0.05, 8.0]), GaussPrior(ci=[0.05, 8.0])],
    theta_dim=2,
    n_samples=400,
    n_chains=2,
    burn_in=200,
    thin=1,
    proposal_scale=0.25,
    positive_params=True,
    jit=True,             # the model term jits; correction stays in Python
    seed=7,
    verbose=True,
    progress=True,
)
mcmc_pkg.run()
mcmc_pkg.summary()

In [ ]:
# Overlay the two posteriors
results_pkg = mcmc_pkg.get_results()
samples_pkg = results_pkg['particles']
chains_pkg = results_pkg['chains']

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))

# Marginal: theta1
axes[0].hist(samples[:, 0], bins=35, density=True, alpha=0.55,
             color='steelblue', edgecolor='white', label='manual')
axes[0].hist(samples_pkg[:, 0], bins=35, density=True, alpha=0.55,
             color='darkorange', edgecolor='white', label='bffg_log_prob')
axes[0].axvline(TRUE_THETA1, color='red', ls='--', label='truth')
axes[0].set(xlabel=r'$\theta_1$', ylabel='density', title=r'Posterior $\theta_1$')
axes[0].legend()

# Marginal: theta2
axes[1].hist(samples[:, 1], bins=35, density=True, alpha=0.55,
             color='steelblue', edgecolor='white', label='manual')
axes[1].hist(samples_pkg[:, 1], bins=35, density=True, alpha=0.55,
             color='darkorange', edgecolor='white', label='bffg_log_prob')
axes[1].axvline(TRUE_THETA2, color='red', ls='--', label='truth')
axes[1].set(xlabel=r'$\theta_2$', ylabel='density', title=r'Posterior $\theta_2$')
axes[1].legend()

# Joint scatter
axes[2].scatter(samples[:, 0], samples[:, 1], s=4, alpha=0.25,
                color='steelblue', label='manual')
axes[2].scatter(samples_pkg[:, 0], samples_pkg[:, 1], s=4, alpha=0.25,
                color='darkorange', label='bffg_log_prob')
axes[2].scatter([TRUE_THETA1], [TRUE_THETA2], color='red', marker='*', s=200,
                edgecolor='black', zorder=5, label='truth')
axes[2].set(xlabel=r'$\theta_1$', ylabel=r'$\theta_2$', title='Joint posterior')
axes[2].legend()

plt.tight_layout(); plt.show()

print("Posterior means:")
print(f"  manual:        theta1 = {results['theta_mean'][0]:.3f}, "
      f"theta2 = {results['theta_mean'][1]:.3f}")
print(f"  bffg_log_prob: theta1 = {results_pkg['theta_mean'][0]:.3f}, "
      f"theta2 = {results_pkg['theta_mean'][1]:.3f}")
print(f"  truth:         theta1 = {TRUE_THETA1}, theta2 = {TRUE_THETA2}")

In [ ]:
# Per-evaluation comparison: stack repeated log-likelihood evaluations at the truth
# to compare estimator variance between the two BFFG implementations.
n_repeats = 30

ll_manual = np.array([
    bffg_log_likelihood(true_theta, observed_data, n_paths=20)
    for _ in range(n_repeats)
])

# Package version: model is deterministic given theta, correction is stochastic
probs_truth, _ = model_pkg(jnp.array(true_theta), jnp.array(observed_data, dtype=jnp.int32))
log_p_model = float(jnp.sum(jnp.log(probs_truth + 1e-30)))
ll_pkg = np.array([
    log_p_model + float(correction_pkg(jnp.array(true_theta)))
    for _ in range(n_repeats)
])

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(ll_manual, bins=15, alpha=0.6, color='steelblue', edgecolor='white',
        label=f'manual (sd={ll_manual.std():.2f})')
ax.hist(ll_pkg, bins=15, alpha=0.6, color='darkorange', edgecolor='white',
        label=f'bffg_log_prob (sd={ll_pkg.std():.2f})')
ax.set(xlabel='BFFG log-likelihood at truth', ylabel='count',
       title=f'Estimator variance ({n_repeats} repeats, n_paths=20 each)')
ax.legend(); plt.tight_layout(); plt.show()

print(f"Manual:        mean = {ll_manual.mean():.2f}, sd = {ll_manual.std():.2f}")
print(f"bffg_log_prob: mean = {ll_pkg.mean():.2f}, sd = {ll_pkg.std():.2f}")

## 11. Posterior predictive check

Draw a posterior parameter sample, simulate fresh observations under it, compare the implied SFS counts to those observed.

In [ ]:
n_pred = 30
pred_counts = np.zeros((n_pred, N_SAMPLES - 1), dtype=int)
post_idx = rng.choice(samples.shape[0], size=n_pred, replace=False)
for k, idx in enumerate(post_idx):
    th = np.asarray(samples[idx])
    for _ in range(N_LOCI):
        try:
            term, _ = simulate_observation(
                jg_cont, theta_proposal, th, THETA_0,
                EPOCH_BOUNDARY, N_SAMPLES, valid_t_vertices,
            )
            j = term_to_type.get(term)
            if j is not None:
                pred_counts[k, j - 1] += 1
        except RuntimeError:
            continue

fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(N_SAMPLES - 1)
ax.boxplot([pred_counts[:, j] for j in x], positions=x, widths=0.5, patch_artist=True,
           boxprops=dict(facecolor='lightsteelblue'))
ax.scatter(x, mut_type_counts, color='red', zorder=5, label='observed', s=60)
ax.set_xticks(x)
ax.set_xticklabels([f'{j+1}-ton' for j in x])
ax.set(ylabel='count per locus-set', title='Posterior predictive SFS counts')
ax.legend(); plt.tight_layout(); plt.show()